## Static pods & `kubeadm` — bootstrapping a cluster

### Static pods

A **static pod** is one the kubelet runs directly from a manifest on disk — *no API server involved*. The kubelet watches `/etc/kubernetes/manifests`, and for each file: reads it, starts the pod, creates a **mirror pod** in the API server so `kubectl get pods` sees it (named `<pod>-<node>`), and keeps it alive.

**You can't `kubectl delete` a static pod** — deleting the mirror just removes the API-server view; the kubelet recreates it in seconds. To stop one, **move or delete its manifest file** on the node.

Why it exists: the control plane's chicken-and-egg — you need the API server to start the API server. Static pods break it. On a kubeadm control-plane node, `/etc/kubernetes/manifests/` holds `etcd.yaml`, `kube-apiserver.yaml`, `kube-controller-manager.yaml`, `kube-scheduler.yaml`. The kubelet boots, reads them, and brings up the control plane — **etcd first**, then the API server, then the rest.

### `kubeadm`

The canonical installer. It **doesn't** provision machines, install the OS, or pick a CNI — those are yours. Given Linux machines with a runtime + the `kubelet`/`kubeadm`/`kubectl` binaries, a typical install is **two commands**:

- **`kubeadm init`** on the first control-plane node: pre-flight checks → generate the PKI in `/etc/kubernetes/pki/` → write the static-pod manifests → wait for the control plane → install CoreDNS + kube-proxy → print a `kubeadm join` command with a token.
- **`kubeadm join`** on every other node: the kubelet uses the token to auth, verifies the CA against a SHA-256 hash, registers the Node.

**A CNI is your job** — until you `kubectl apply` one (e.g. Calico), pods stay `Pending`. Tokens expire in 24h (`kubeadm token create --print-join-command`). On our map static pods and kubeadm build the entire **Control Plane** box from bare metal.